# Régression - Performances en powerlifting

## Import des librairies :

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import validation_curve, RandomizedSearchCV, GridSearchCV
from autogluon.tabular import TabularPredictor
import xgboost as xgb
import optuna

## Pré-traitements sur les données :

##### Chargement des données d'entraînement et de test :

In [62]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

In [63]:
# Nombre de valeurs manquantes par colonne (train puis test) :
print(data_train.isna().sum())
print(data_test.isna().sum())

id                  0
Age                 0
BodyweightKg        0
MeetYear            0
Sex                 0
Equipment           0
AgeClass            0
BirthYearClass      0
WeightClassKg       0
Tested              0
Country             0
State               0
Federation          0
ParentFederation    0
MeetCountry         0
MeetState           0
Sanctioned          0
TotalKg             0
dtype: int64
id                  0
Age                 0
BodyweightKg        0
MeetYear            0
Sex                 0
Equipment           0
AgeClass            0
BirthYearClass      0
WeightClassKg       0
Tested              0
Country             0
State               0
Federation          0
ParentFederation    0
MeetCountry         0
MeetState           0
Sanctioned          0
dtype: int64


Il n'y a aucune valeur manquante (NaN) dans le dataset d'entraînement (`data_train`) ni dans le dataset de test (`data_test`).

### Variables continues :

Rien ne change pour ces variables ci : `Age`, `BodyweightKg`, `Meet_year`

### Variables catégorielles : 

Il y a des variables ordinales, pour lesquelles l'ordre est important : `AgeClass`, `BirthYearClass`, `WeightClassKg`

Il y a des variables nominales (pas forcément textuelles !), pour lesquelles l'ordre ne compte pas : `Sex`, `Equipment`, `Tested`, `Country`,  `State`, `Federation`, `ParentFederation`, `MeetCountry`, `MeetState`, `Sanctioned`

Ces deux types de variables ne se traitent pas avec le même encodeur. On sépare les deux groupes :

##### Variables catégorielles ordinales :

Pour `AgeClass`:

In [64]:
# Chaque tranche est remplacée par son milieu (ex: "24-34" -> 29) pour garder l'ordre.
# "Unknown" -> -2
ordre_AgeClass = {"16-17": 16, "24-34": 29, "50-54": 52, "20-23": 21, "18-19": 18,
             "55-59": 57, "40-44": 42, "35-39": 37, "60-64": 62, 
             "13-15": 14, "45-49": 47, "5-12": 9, "65-69": 67, 
             "70-74": 72, "80-84": 82, "Unknown": -2, "75-79": 77,
             "85-89": 87, "90-999": 95}

data_train["AgeClass"] = data_train["AgeClass"].map(ordre_AgeClass)
data_test["AgeClass"] = data_test["AgeClass"].map(ordre_AgeClass)

Pour `BirthYearClass`:

In [65]:
ordre_BirthYearClass = {"14-18": 16, "24-39": 31, "50-59": 54, "19-23": 21, "40-49": 44, "Unknown": -2, "60-69": 64, "70-999": 70}

data_train["BirthYearClass"] = data_train["BirthYearClass"].map(ordre_BirthYearClass)
data_test["BirthYearClass"] = data_test["BirthYearClass"].map(ordre_BirthYearClass)

Pour `WeightClassKg` :

In [66]:
ordre_WeightClassKg = {
    # --- Inconnus / parasites ---
    "Unknown": -2, "+": -2,

    # --- Valeurs numériques X ---
    "30": 30, "31": 31, "33": 33, "34.9": 34.9, "35": 35, "36": 36, "36.2": 36.2,
    "40": 40, "43": 43, "43.5": 43.5, "44": 44, "44.2": 44.2, "44.5": 44.5,
    "46": 46, "47": 47, "47.5": 47.5, "47.6": 47.6, "47.8": 47.8, "48": 48,
    "49": 49, "50": 50, "50.5": 50.5, "51.7": 51.7, "51.9": 51.9, "52": 52,
    "53": 53, "54": 54, "54.4": 54.4, "55": 55, "55.5": 55.5, "55.7": 55.7,
    "56": 56, "56.5": 56.5, "56.7": 56.7, "57": 57, "58": 58, "58.5": 58.5,
    "59": 59, "59.8": 59.8, "60": 60, "60.1": 60.1, "62": 62, "63": 63,
    "63.5": 63.5, "64": 64, "65": 65, "66": 66, "67.1": 67.1, "67.3": 67.3,
    "67.5": 67.5, "69": 69, "70": 70, "70.3": 70.3, "72": 72, "72.5": 72.5,
    "73": 73, "74": 74, "74.8": 74.8, "75": 75, "76": 76, "77": 77,
    "77.5": 77.5, "78.9": 78.9, "79.5": 79.5, "80": 80, "81.5": 81.5,
    "81.6": 81.6, "82.1": 82.1, "82.3": 82.3, "82.5": 82.5, "83": 83,
    "83.9": 83.9, "84": 84, "85": 85, "89.8": 89.8, "90": 90, "90.7": 90.7,
    "91.1": 91.1, "93": 93, "-93": 93, "93.8": 93.8, "94": 94, "95": 95,
    "99.7": 99.7, "100": 100, "102": 102, "104.7": 104.7, "105": 105,
    "109": 109, "109.7": 109.7, "110": 110, "113.4": 113.4, "113.5": 113.5,
    "115": 115, "117.5": 117.5, "117.7": 117.7, "118": 118, "120": 120,
    "120.2": 120.2, "124.7": 124.7, "125": 125, "127": 127, "135.9": 135.9,
    "136": 136, "136.5": 136.5, "139.7": 139.7, "140": 140, "143": 143,
    "145": 145, "148": 148, "155": 155,

    # --- Catégories "X+"  ---
    # Les catégories "X+" (ex: "84+") valent X + 0.01 : juste au-dessus de la classe X, pour conserver l'ordre.
    "56+": 56.01, "57+": 57.01, "60+": 60.01, "63+": 63.01, "67.5+": 67.51,
    "69+": 69.01, "72+": 72.01, "74+": 74.01, "75+": 75.01, "76+": 76.01,
    "80+": 80.01, "81.5+": 81.51, "82.5+": 82.51, "83+": 83.01, "84+": 84.01,
    "85+": 85.01, "90+": 90.01, "90.7+": 90.71, "93+": 93.01, "100+": 100.01,
    "105+": 105.01, "109+": 109.01, "110+": 110.01, "117.7+": 117.71,
    "120+": 120.01, "120.2+": 120.21, "124.7+": 124.71, "125+": 125.01,
    "139.7+": 139.71, "140+": 140.01, "143+": 143.01, "145+": 145.01,
    "155+": 155.01,

    # --- Valeurs présentes dans data_test, mais absentes de data_train ---
    "52.5": 52.5, "57.5": 57.5, "79.3": 79.3, "101.6": 101.6, "103": 103, "181": 181,
    "117.5+": 117.51,
}

data_train["WeightClassKg"] = data_train["WeightClassKg"].map(ordre_WeightClassKg)
data_test["WeightClassKg"] = data_test["WeightClassKg"].map(ordre_WeightClassKg)

##### Variables catégorielles nominales :

In [67]:
cols_nom = [
    "Sex", "Equipment",
    "Tested", "Country",
    "State", "Federation", "ParentFederation", "MeetCountry",
    "MeetState", "Sanctioned"]

enc_nom = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2) 
# unknown_value=-1 : catégorie vue dans le test mais pas dans le train
# encoded_missing_value=-2 : vraie valeur manquante (NaN)
# fit sur le train uniquement, puis transform sur le test (évite la fuite de données)
data_train[cols_nom] = enc_nom.fit_transform(data_train[cols_nom])
data_test[cols_nom] = enc_nom.transform(data_test[cols_nom])

On enregistre ces nouveaux datasets :

In [68]:
data_train.to_csv("data_train_clean.csv", index=True)
data_test.to_csv("data_test_clean.csv", index=True)

## Modèles prédictifs :

##### Chargements des données d'entraînement et de test nettoyées :

In [69]:
data_train = pd.read_csv("data_train_clean.csv")
data_test = pd.read_csv("data_test_clean.csv")

data_train = data_train.drop(columns=["Unnamed: 0"])
data_test = data_test.drop(columns=["Unnamed: 0"])

##### On sépare les covariables de la variable cible dans les données d'entraînement :

In [70]:
X_train = data_train.drop(columns=["TotalKg", "id"])
Y_train = data_train["TotalKg"]
X_test = data_test.drop(columns=["id"])

On normalise :

In [71]:
scaler = StandardScaler()
X_train_normed = scaler.fit_transform(X_train)
X_test_normed = scaler.transform(X_test)

# Nécessaire pour la régression linéaire et le KNN
# Inutile pour les modèles à base d'arbres

### Régression linéaire :

##### Entraînement :

In [72]:
model = LinearRegression()
model.fit(X_train_normed, Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](16,)","[-28.39, 75.58,-14.41,..., 0.27, 4.64, 1.75]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,493
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,16
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(16)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](16,)","[565.2 ,544.89,470.35,...,112.93,112.49, 38.81]"


##### Prédictions :

In [73]:
y_pred = model.predict(X_test_normed)

submission = pd.DataFrame({
    "id": data_test["id"],
    "TotalKg": y_pred   
})
submission.to_csv("regression_submission_linear_regression.csv", index=False)

### K plus proches voisins :

##### Entraînement :

In [24]:
knn_model = KNeighborsRegressor()
knn_model.fit(X_train_normed, Y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"effective_metric_ effective_metric_: str or callableThe distance metric to use. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
"effective_metric_params_ effective_metric_params_: dictAdditional keyword arguments for the metric function. For most metricswill be same with `metric_params` parameter, but may also contain the`p` parameter value if the `effective_metric_` attribute is set to'minkowski'.",dict,{}


##### Prédictions :

In [ ]:
y_pred = knn_model.predict(X_test_normed)

submission = pd.DataFrame({
    "id": data_test["id"],
    "TotalKg": y_pred   
})
submission.to_csv("regression_submission_knn.csv", index=False)

### Arbres de décision :

##### Recherche d'hyperparamètres et entraînement :

Optimisation de `max_depth` et `min_samples_leaf` :

In [75]:
params = {
    "max_depth": [8, 10, 12, 14, 16, 20, None],
    "min_samples_leaf": [1, 5, 10, 20, 50, 100],
}

search_tree = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    params, cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
search_tree.fit(X_train, Y_train)

print("Meilleurs paramètres :", search_tree.best_params_)
print(f"RMSE : {-search_tree.best_score_:.2f}")

Meilleurs paramètres : {'max_depth': 14, 'min_samples_leaf': 50}
RMSE CV : 87.06


##### Prédictions :

In [77]:
best_tree = search_tree.best_estimator_
y_pred = best_tree.predict(X_test)

submission = pd.DataFrame({
    "id": data_test["id"],
    "TotalKg": y_pred   
})
submission.to_csv("regression_submission_decision_tree_searchfit.csv", index=False)

### Forêt aléatoire : 

##### Recherche d'hyperparamètres et entraînement :

Optimisation de `n_estimator`, `max_depth`, `min_samples_leaf`, `max_features` et `max_samples` :

In [ ]:
params = {
    "n_estimators": [200, 400, 600],
    "max_depth": [10, 15, 20, None],
    "min_samples_leaf": [5, 10, 20, 50],
    "max_features": [0.3, 0.5, 0.7, 1.0],   
    "max_samples": [0.5, 0.7, 0.9, None],   
}

search_rf = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=-1, random_state=42),
    params, n_iter=15, cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,          
    random_state=42,
)
search_rf.fit(X_train, Y_train)

print("Meilleurs paramètres :", search_rf.best_params_)
print(f"RMSE CV : {-search_rf.best_score_:.2f}")

##### Prédictions :

In [ ]:
best_rf = search_rf.best_estimator_
y_pred = best_rf.predict(X_test)

submission = pd.DataFrame({
    "id": data_test["id"],
    "TotalKg": y_pred   
})
submission.to_csv("regression_submission_random_forest_searchfit.csv", index=False)

### Gradient Boosting :

#### `HistGradientBoostingRegressor` :

##### Recherche d'hyperparamètres et entraînement :

Optimisation de `max_iter`, `learning_rate`, `max_depth`, `max_leaf_nodes`, `min_samples_leaf` et `l2_regularization` :

In [ ]:
params = {
    "max_iter": [200, 400, 600, 1000],       
    "learning_rate": [0.02, 0.05, 0.1],
    "max_depth": [None, 4, 6, 8],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 50, 100],
    "l2_regularization": [0, 0.1, 1, 10],
}

search_hgb = RandomizedSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    params, n_iter=20, cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=42,
)
search_hgb.fit(X_train, Y_train)

print("Meilleurs paramètres :", search_hgb.best_params_)
print(f"RMSE CV : {-search_hgb.best_score_:.2f}")

##### Prédictions :

In [ ]:
best_hgb = search_hgb.best_estimator_
y_pred = best_hgb.predict(X_test)

submission = pd.DataFrame({
    "id": data_test["id"],
    "TotalKg": y_pred   
})
submission.to_csv("regression_submission_gradient_boosting_searchfit.csv", index=False)

#### `XGBoost` et `Optuna` :

##### Recherche d'hyperparamètres et entraînement :

Optimisation de `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_lambda` et `reg_alpha` :

In [ ]:
dtrain = xgb.DMatrix(X_train, label=Y_train)

DEVICE = "cuda"   # mettre "cpu" si pas de GPU

def objective(trial):
    params = {
        "objective": "reg:squarederror", "eval_metric": "rmse",
        "tree_method": "hist", "device": DEVICE,
        "eta": 0.05,
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 100, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
    }
    cv = xgb.cv(params, dtrain, num_boost_round=5000, nfold=5, seed=0,
                early_stopping_rounds=100, verbose_eval=False)
    trial.set_user_attr("best_iter", len(cv))
    return cv["test-rmse-mean"].iloc[-1]

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=30)

print("Meilleur RMSE :", study.best_value)
print("Meilleurs paramètres :", study.best_params)

# Modèle final sur tout le train
best = study.best_params | {"objective": "reg:squarederror", "eval_metric": "rmse",
                            "tree_method": "hist", "device": DEVICE, "eta": 0.05}
n_iter = int(study.best_trial.user_attrs["best_iter"] * 1.1)
final = xgb.train(best, dtrain, num_boost_round=n_iter)

##### Prédictions :

In [ ]:
y_pred = final.predict(xgb.DMatrix(X_test))

submission = pd.DataFrame({
    "id": data_test["id"],
    "TotalKg": y_pred
})
submission.to_csv("regression_submission_xgb_optuna.csv", index=False)

### AutoGluon :

In [ ]:
predictor = TabularPredictor(
    label="TotalKg",
    eval_metric="root_mean_squared_error",       
).fit(
    data_train.drop(columns=["id"]),
    presets="best",
    time_limit=300,             
    num_gpus=1,                  
)

##### Prédictions :

In [ ]:
y_pred = predictor.predict(data_test.drop(columns=["id"]))

submission = pd.DataFrame({
    "id": data_test["id"],
    "TotalKg": y_pred    
})
submission.to_csv("regression_submission_auto_gluon.csv", index=False)